# 06 Ejemplos de predicción

Este notebook usa los modelos guardados por `05-final-training-and-test.ipynb` en `results/models/`. Ejecuta primero el notebook 05.

Cada consulta debe incluir las 22 variables definidas en `notebooks/features.py`. Estas predicciones son ejemplos de uso; no constituyen una evaluación del modelo.

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from features import FEATURES

OUT = Path("..") / "results" / "models"

## 1. Clasificar un ingreso nuevo

El pipeline guardado ya incluye el preprocesamiento. La fila debe tener las mismas 22 variables derivadas que `FEATURES` (no las columnas originales del CSV); cambia los valores de ejemplo por los del animal que quieras consultar. Las categorías desconocidas se gestionan con el preprocesador.

In [ ]:
modelo_clf = joblib.load(OUT / "model_clf.joblib")

EJEMPLO = {
    "animal_type": "Dog", "intake_type": "Stray", "intake_condition": "Normal",
    "sex": "Female", "weekday": "Tuesday", "season": "autumn",
    "breed_main": "Border Terrier", "color_main": "White", "city": "Austin",
    "is_neutered": 1, "is_mix": 0, "is_crossbreed": 1, "is_bicolor": 1,
    "is_tabby": 0, "has_location_detail": 1, "is_baby": 0, "condition_serious": 0,
    "age_days": 2555, "prior_stays": 0, "hour": 7, "month": 10, "t_months": 10,
}

consulta_r = pd.DataFrame([EJEMPLO], columns=FEATURES)
prediccion_r = modelo_clf.predict(consulta_r)[0]
probabilidades_r = pd.Series(
    modelo_clf.predict_proba(consulta_r)[0], index=modelo_clf.classes_, name="probabilidad estimada"
).sort_values(ascending=False)

print("Resultado predicho:", prediccion_r)
print(probabilidades_r.to_frame().round(3))

## 2. Estimar los días hasta la adopción

La regresión se entrenó solo con animales adoptados. Por eso el resultado se interpreta como los días estimados **si el animal es adoptado**; no es una estimación para Transfer, Return to Owner, Euthanasia u Other. Se usa el mismo ejemplo y las mismas variables.

In [ ]:
modelo_reg = joblib.load(OUT / "model_reg.joblib")
consulta_reg = pd.DataFrame([EJEMPLO], columns=FEATURES)

# El modelo predice log(1 + días); np.expm1 lo devuelve a días.
prediccion_log = modelo_reg.predict(consulta_reg)[0]
dias_estimados = np.expm1(prediccion_log)
print(f"Días estimados en el refugio, si es adoptado: {dias_estimados:.1f}")